# crypto_4h_strat_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (189).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `crypto` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 8 |
| Detected functions | resample_1hr_custom, calculate_strategy_signals, prepare_backtest_data, backtest_single_ticker, run_backtest |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T5_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to Datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the Date range
start_Date = '2021-01-01'
end_Date = '2025-12-30'

# Define the Tickers you want to filter
selected_Tickers = ['BTCUSDT', 'SOLUSDT', 'XRPUSDT', 'BNBUSDT', 'ETHUSDT']  # Add Tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_Date) &
    (data["Date"] <= end_Date) &
    (data["Ticker"].isin(selected_Tickers))
]

# Display filtered results
data_1min = data
data

In [ ]:
import pandas as pd

# Ensure Date is Datetime
data_1min['Date'] = pd.to_datetime(data_1min['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    Ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
    }).dropna().reset_index()
    resampled['Ticker'] = Ticker
    return resampled

# Apply per Ticker
data_1hr = data_1min.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

### **INDICATOR CODE**

In [ ]:
"""
Cryptocurrency Long-Only Trading Strategy - CONVEX VERSION
==========================================================
A momentum-based trading system focused on rotation ignition:
- Beta-adjusted returns vs BTC
- Velocity and acceleration confirmation
- PCA-based curvature as PRIMARY GATEKEEPER
- Hard stops only (no trailing stops that kill convexity)
"""

import pandas as pd
import numpy as np
from numba import njit
from tqdm import tqdm
from sklearn.decomposition import PCA


# ============================================================================
# TECHNICAL INDICATORS
# ============================================================================

def calculate_strategy_signals(
    df: pd.DataFrame,
    altcoin_tickers: list,
    btc_ticker: str = 'BTC',
    beta_lookback: int = 180,
    velocity_period: int = 12,
    acceleration_norm_window: int = 84,
    pca_n_components: int = 3,
    curvature_lookback: int = 168,
    curvature_entry_percentile: float = 95.0,  # Changed to float for precision
    acceleration_entry_threshold: float = 0.3,
    acceleration_exit_threshold: float = -0.3,
    warmup_period: int = 180,
    curvature_spike_multiplier: float = 1.25  # NEW: Require spike, not just level
) -> pd.DataFrame:
    """
    Calculate trading signals with CURVATURE as primary gatekeeper.

    Strategy Logic (CONVEX):
    1. Calculate beta-adjusted returns (residuals vs BTC)
    2. Compute velocity and acceleration
    3. Use PCA to calculate curvature
    4. ENTRY: Curvature SPIKE above percentile threshold + acceleration confirmation
    5. EXIT: Acceleration reversal OR time stop

    Key Changes:
    - Curvature must SPIKE (not just exceed threshold)
    - Acceleration only confirms at entry/exit, doesn't drive
    - No trailing stops (preserves convexity)

    Args:
        df: Price data with columns [date, Ticker, close, high, low]
        altcoin_tickers: List of altcoin symbols to trade
        btc_ticker: Bitcoin reference ticker
        beta_lookback: Window for beta calculation
        velocity_period: Lookback for velocity calculation
        acceleration_norm_window: Window for acceleration normalization
        pca_n_components: Number of PCA components
        curvature_lookback: Window for curvature percentile
        curvature_entry_percentile: Percentile threshold (92.5-97.5 recommended)
        acceleration_entry_threshold: Min acceleration for entry confirmation
        acceleration_exit_threshold: Max acceleration for exit
        warmup_period: Initial bars to ignore for signals
        curvature_spike_multiplier: Multiplier for spike detection (default 1.5)

    Returns:
        DataFrame with 'signal' column (1=entry, -1=exit, 0=hold)
    """
    df = df.copy()

    # Ensure datetime format
    if 'Date' in df.columns:
        df['date'] = pd.to_datetime(df['Date'])
    elif 'date' not in df.columns:
        raise ValueError("DataFrame must contain 'Date' or 'date' column")

    # Pivot prices to wide format for vectorized calculations
    prices = (
        df.pivot(index="date", columns="Ticker", values="close")
          .ffill()
          .bfill()
          .sort_index()
    )

    # Calculate log returns
    log_returns = np.log(prices).diff()
    btc_returns = log_returns[btc_ticker]

    # Calculate rolling beta (covariance / variance)
    covariance = log_returns[altcoin_tickers].rolling(beta_lookback).cov(btc_returns)
    variance = btc_returns.rolling(beta_lookback).var()
    beta = covariance.div(variance, axis=0).ffill().fillna(1.0)

    # Calculate beta-adjusted residuals
    residuals = log_returns[altcoin_tickers] - beta.mul(btc_returns, axis=0)
    residuals = residuals.dropna()

    # Velocity (first derivative) and acceleration (second derivative)
    velocity = residuals.diff(velocity_period)
    acceleration = velocity.diff(velocity_period)

    # Normalize acceleration (z-score)
    acceleration_mean = acceleration.rolling(
        acceleration_norm_window, min_periods=10
    ).mean()
    acceleration_std = acceleration.rolling(
        acceleration_norm_window, min_periods=10
    ).std()
    normalized_acceleration = (acceleration - acceleration_mean) / (acceleration_std + 1e-8)

    # Calculate curvature using PCA
    valid_rows = ~(velocity.isna().any(axis=1) | acceleration.isna().any(axis=1))
    curvature = pd.Series(0.0, index=velocity.index)

    if valid_rows.sum() >= 10:
        n_components = min(pca_n_components, velocity.shape[1])
        pca = PCA(n_components=n_components)
        pca.fit(velocity.loc[valid_rows].fillna(0))

        # Transform to PCA space
        velocity_pca = pca.transform(velocity.fillna(0))
        acceleration_pca = pca.transform(acceleration.fillna(0))

        # Pad if necessary (for cross product calculation)
        if velocity_pca.shape[1] < 3:
            pad_width = 3 - velocity_pca.shape[1]
            velocity_pca = np.pad(velocity_pca, ((0, 0), (0, pad_width)))
            acceleration_pca = np.pad(acceleration_pca, ((0, 0), (0, pad_width)))

        # Curvature = ||v × a|| / ||v||^3
        velocity_norm = np.linalg.norm(velocity_pca, axis=1)
        cross_product_norm = np.linalg.norm(np.cross(velocity_pca, acceleration_pca), axis=1)
        valid_mask = velocity_norm > 1e-8
        curvature.values[valid_mask] = cross_product_norm[valid_mask] / (
            velocity_norm[valid_mask] ** 3 + 1e-8
        )

    # Calculate curvature threshold (rolling percentile)
    curvature_threshold = curvature.rolling(
        curvature_lookback, min_periods=20
    ).quantile(curvature_entry_percentile / 100.0)

    # NEW: Detect curvature SPIKES (not just levels)
    # A spike means curvature is rising rapidly and exceeds threshold
    curvature_prev = curvature.shift(1)
    curvature_change = curvature - curvature_prev
    curvature_is_spiking = (
        (curvature >= curvature_threshold) &
        (curvature > curvature_prev * curvature_spike_multiplier)
    )

    # Generate signals (LONG ONLY)
    max_acceleration = normalized_acceleration.max(axis=1)
    min_acceleration = normalized_acceleration.min(axis=1)

    # ENTRY: Curvature spike + acceleration confirmation
    entry_condition = (
        curvature_is_spiking &
        (max_acceleration > acceleration_entry_threshold)
    )

    # EXIT: Acceleration reversal only (no trailing stops)
    exit_condition = min_acceleration < acceleration_exit_threshold

    signal = pd.Series(0, index=normalized_acceleration.index, dtype=np.int8)
    signal[entry_condition] = 1   # LONG entry
    signal[exit_condition] = -1   # Exit signal

    # Create signal DataFrame
    signal_df = pd.DataFrame({
        'date': signal.index,
        'signal': signal.values,
        'curvature': curvature.values,
        'curvature_threshold': curvature_threshold.values,
        'max_accel': max_acceleration.values
    })

    # Merge signals back to original data
    result = df.merge(signal_df, on='date', how='left')
    result['signal'] = result['signal'].fillna(0).astype(int)

    # Apply warmup period (ignore early signals per ticker)
    if warmup_period > 0:
        for ticker in result['Ticker'].unique():
            ticker_mask = result['Ticker'] == ticker
            ticker_data = result[ticker_mask]
            if len(ticker_data) > warmup_period:
                warmup_indices = ticker_data.index[:warmup_period]
                result.loc[warmup_indices, 'signal'] = 0

    return result


def prepare_backtest_data(
    data: pd.DataFrame,
    altcoin_tickers: list = None,
    btc_ticker: str = 'BTC',
    curvature_entry_percentile: float = 95.0
) -> pd.DataFrame:
    """
    Prepare data with indicators and signals, properly shifted to avoid look-ahead bias.

    Args:
        data: Raw OHLC data with columns [Date, Ticker, open, high, low, close]
        altcoin_tickers: List of altcoin tickers (default: all except BTC)
        btc_ticker: Bitcoin ticker symbol
        curvature_entry_percentile: Percentile for curvature gating (92.5-97.5)

    Returns:
        Clean DataFrame ready for backtesting
    """
    data = data.copy()

    # Standardize column names
    if 'Date' in data.columns:
        data['date'] = pd.to_datetime(data['Date'])

    # Determine altcoin tickers if not provided
    if altcoin_tickers is None:
        all_tickers = data['Ticker'].unique()
        altcoin_tickers = [t for t in all_tickers if t != btc_ticker]

    # Calculate indicators and signals
    data = calculate_strategy_signals(
        data,
        altcoin_tickers=altcoin_tickers,
        btc_ticker=btc_ticker,
        curvature_entry_percentile=curvature_entry_percentile
    )

    # Shift signals to avoid look-ahead bias
    # (we can only act on information available at the open of the next bar)
    data['signal'] = data.groupby('Ticker')['signal'].shift(1)

    # Remove rows with missing signals
    data = data.dropna(subset=['signal'])
    data['signal'] = data['signal'].astype(int)

    return data[['date', 'Ticker', 'open', 'high', 'low', 'close',
                 'signal', 'curvature', 'curvature_threshold', 'max_accel']].reset_index(drop=True)


# ============================================================================
# BACKTESTING ENGINE (CONVEX VERSION)
# ============================================================================

@njit
def backtest_single_ticker(
    opens: np.ndarray,
    highs: np.ndarray,
    lows: np.ndarray,
    closes: np.ndarray,
    signals: np.ndarray,
    initial_capital: float,
    fee_rate: float,
    slippage_rate: float,
    hard_stop_loss_pct: float,
    max_holding_bars: int
):
    """
    Numba-compiled backtest for a single ticker (CONVEX VERSION).

    Position Management:
    - Enter long on signal=1
    - Exit on: (1) signal=-1, (2) hard stop hit, (3) time stop
    - NO TRAILING STOPS (preserves convexity)

    Args:
        opens, highs, lows, closes: OHLC price arrays
        signals: Trading signals (1=entry, -1=exit, 0=hold)
        initial_capital: Starting capital per position
        fee_rate: Trading fee as decimal (e.g., 0.001 = 0.1%)
        slippage_rate: Slippage as decimal
        hard_stop_loss_pct: Hard stop loss percentage (catastrophic protection only)
        max_holding_bars: Maximum bars to hold position (time stop)

    Returns:
        Tuple of arrays: (entry_idx, exit_idx, entry_prices, exit_prices,
                         quantities, pnls, exit_reasons)
    """
    n = len(opens)

    # Pre-allocate result arrays
    entry_indices = []
    exit_indices = []
    entry_price_list = []
    exit_price_list = []
    quantity_list = []
    pnl_list = []
    exit_reason_list = []

    # Position state
    has_position = False
    entry_price = 0.0
    position_size = 0.0
    hard_stop_level = 0.0
    entry_bar_idx = 0
    bars_in_position = 0

    for i in range(n):
        if not has_position:
            # Look for entry signal (LONG only)
            if signals[i] == 1:
                entry_price = opens[i] * (1 + slippage_rate)
                position_size = initial_capital / entry_price
                hard_stop_level = entry_price * (1 - hard_stop_loss_pct)

                # Check if hard stop is hit on entry bar
                if lows[i] <= hard_stop_level:
                    exit_price = hard_stop_level * (1 - slippage_rate)
                    gross_pnl = (exit_price - entry_price) * position_size
                    fees = fee_rate * (entry_price + exit_price) * position_size
                    net_pnl = gross_pnl - fees

                    entry_indices.append(i)
                    exit_indices.append(i)
                    entry_price_list.append(entry_price)
                    exit_price_list.append(exit_price)
                    quantity_list.append(position_size)
                    pnl_list.append(net_pnl)
                    exit_reason_list.append(0)  # HARD_STOP_LOSS
                    continue

                # Open position
                has_position = True
                entry_bar_idx = i
                bars_in_position = 0

        else:
            bars_in_position += 1

            # Check for exit signal first
            if signals[i] == -1:
                exit_price = opens[i] * (1 - slippage_rate)
                gross_pnl = (exit_price - entry_price) * position_size
                fees = fee_rate * (entry_price + exit_price) * position_size
                net_pnl = gross_pnl - fees

                entry_indices.append(entry_bar_idx)
                exit_indices.append(i)
                entry_price_list.append(entry_price)
                exit_price_list.append(exit_price)
                quantity_list.append(position_size)
                pnl_list.append(net_pnl)
                exit_reason_list.append(1)  # SIGNAL_EXIT
                has_position = False
                continue

            # Check hard stop (catastrophic protection only)
            if lows[i] <= hard_stop_level:
                exit_price = hard_stop_level * (1 - slippage_rate)
                gross_pnl = (exit_price - entry_price) * position_size
                fees = fee_rate * (entry_price + exit_price) * position_size
                net_pnl = gross_pnl - fees

                entry_indices.append(entry_bar_idx)
                exit_indices.append(i)
                entry_price_list.append(entry_price)
                exit_price_list.append(exit_price)
                quantity_list.append(position_size)
                pnl_list.append(net_pnl)
                exit_reason_list.append(0)  # HARD_STOP_LOSS
                has_position = False
                continue

            # Check time stop
            if bars_in_position >= max_holding_bars:
                exit_price = closes[i] * (1 - slippage_rate)
                gross_pnl = (exit_price - entry_price) * position_size
                fees = fee_rate * (entry_price + exit_price) * position_size
                net_pnl = gross_pnl - fees

                entry_indices.append(entry_bar_idx)
                exit_indices.append(i)
                entry_price_list.append(entry_price)
                exit_price_list.append(exit_price)
                quantity_list.append(position_size)
                pnl_list.append(net_pnl)
                exit_reason_list.append(2)  # TIME_STOP
                has_position = False

    return (entry_indices, exit_indices, entry_price_list, exit_price_list,
            quantity_list, pnl_list, exit_reason_list)


def run_backtest(
    data: pd.DataFrame,
    initial_capital: float = 100000,
    fee_rate: float = 0.00025,
    slippage_rate: float = 0.001,
    hard_stop_loss_pct: float = 0.15,  # Wider hard stop (catastrophic only)
    max_holding_bars: int = 100  # Time stop
) -> pd.DataFrame:
    """
    Run backtest across all tickers using Numba for speed.

    Args:
        data: Prepared DataFrame from prepare_backtest_data()
        initial_capital: Starting capital per position
        fee_rate: Trading fee (0.00025 = 0.025%)
        slippage_rate: Slippage rate (0.001 = 0.1%)
        hard_stop_loss_pct: Hard stop loss (0.15 = 15%, catastrophic protection only)
        max_holding_bars: Maximum bars to hold (time stop)

    Returns:
        DataFrame with trade results (tradebook)
    """
    all_trades = []
    exit_reason_map = {
        0: 'HARD_STOP_LOSS',
        1: 'SIGNAL_EXIT',
        2: 'TIME_STOP'
    }

    for ticker in tqdm(data['Ticker'].unique(), desc="Backtesting"):
        ticker_data = (
            data[data['Ticker'] == ticker]
            .sort_values('date')
            .reset_index(drop=True)
        )

        # Convert to numpy arrays for Numba
        opens = ticker_data['open'].values
        highs = ticker_data['high'].values
        lows = ticker_data['low'].values
        closes = ticker_data['close'].values
        signals = ticker_data['signal'].values
        dates = ticker_data['date'].values

        # Run backtest
        results = backtest_single_ticker(
            opens, highs, lows, closes, signals,
            initial_capital, fee_rate, slippage_rate,
            hard_stop_loss_pct, max_holding_bars
        )

        entry_idx, exit_idx, entry_prices, exit_prices, quantities, pnls, exit_reasons = results

        # Convert results to trade records
        for j in range(len(entry_idx)):
            all_trades.append({
                'ticker': ticker,
                'entry_date': dates[entry_idx[j]],
                'exit_date': dates[exit_idx[j]],
                'entry_price': entry_prices[j],
                'exit_price': exit_prices[j],
                'quantity': quantities[j],
                'direction': 'LONG',
                'pnl': pnls[j],
                'exit_reason': exit_reason_map[exit_reasons[j]]
            })

    if not all_trades:
        return pd.DataFrame()

    # Create tradebook
    trades_df = pd.DataFrame(all_trades)
    trades_df = trades_df.sort_values(['ticker', 'entry_date']).reset_index(drop=True)
    trades_df['cumulative_pnl'] = trades_df.groupby('ticker')['pnl'].cumsum()

    # Format final tradebook
    tradebook = pd.DataFrame({
        'Ticker': trades_df['ticker'],
        'Entry Time': trades_df['entry_date'],
        'Exit Time': trades_df['exit_date'],
        'Entry Price': trades_df['entry_price'],
        'Exit Price': trades_df['exit_price'],
        'Quantity': trades_df['quantity'],
        'PnL': trades_df['pnl'],
        'Cumulative PnL': trades_df['cumulative_pnl'],
        'Direction': trades_df['direction'],
        'Exit Reason': trades_df['exit_reason']
    })

    return tradebook


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

if __name__ == "__main__":
    # Load your data
    # data = pd.read_csv('your_data.csv')

    # Prepare data with indicators and signals
    # Try different percentiles: 92.5, 95.0, 97.5
    prepared_data = prepare_backtest_data(
        data,
        btc_ticker='BTCUSDT',
        curvature_entry_percentile=95.0  # Adjust this
    )

    # Run backtest
    tradebook = run_backtest(
        prepared_data,
        initial_capital=100000,
        fee_rate=0.0003,
        slippage_rate=0.0003,
        hard_stop_loss_pct=0.02,  # Wide hard stop
        max_holding_bars=100  # Time stop
    )

    # Save results
    tradebook.to_csv("/content/drive/MyDrive/tradebook_convex.csv", index=False)

In [ ]:
tradebook